<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/crewai_multiagent_level_1_updated_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Create Multi Agents to Research and Write an Article

In this workshop you will build a **team of AI agents** (Planner, Writer, Editor) that research a topic, write a 1500-word article and edit it, using the **latest CrewAI** and an **OpenAI reasoning model**. At the end you put everything behind a **Gradio app with a public link**.

Live demo of the finished product: https://huggingface.co/spaces/decodingdatascience/fromprompttopost

**What you will build, step by step**

| Step | What we do |
|---|---|
| 1 | Install the latest libraries |
| 2 | Check the installed versions |
| 3 | Import from CrewAI |
| 4 | Add API keys from Colab Secrets |
| 5 | Set up the reasoning model |
| 6 | Test the model |
| 7 | Set up the tool (web search) |
| 8 | Create the Planner agent |
| 9 | Create the Writer agent |
| 10 | Create the Editor agent |
| 11 | Create the Plan task |
| 12 | Create the Write task |
| 13 | Create the Edit task |
| 14 | Assemble the Crew |
| 15 | Run the Crew (async) |
| 16 | Display the final article |
| 17 | Deploy the crew as a Gradio app |
| 18 | Why this matters |

# Step 1: Install the latest libraries

> **Note:** We install the **latest** versions of `crewai`, `crewai-tools` (web search and website tools), `openai` (the model provider) and `gradio` (the web UI). The old pinned versions (`crewai==0.28.8`) are outdated and do not support current models. `-U` upgrades anything already installed. **If Colab shows a "Restart session" button after this cell, click it, then continue from Step 2.**

In [1]:
!pip install -q -U crewai crewai-tools openai gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


# Step 2: Check the installed versions

> **Note:** Always print the versions at the start of a workshop. If something breaks later, this is the first thing to compare. We also switch off warnings so the output stays clean for learners.

In [2]:
import warnings
warnings.filterwarnings("ignore")

from importlib.metadata import version
for pkg in ["crewai", "crewai-tools", "openai", "gradio"]:
    print(f"{pkg:15s} {version(pkg)}")

crewai          1.15.44
crewai-tools    1.15.44
openai          2.54.0
gradio          6.32.0


# Step 3: Import from CrewAI

> **Note:** Four building blocks make a crew. `Agent` is who does the work, `Task` is what needs to be done, `Crew` is the team that runs the tasks, and `LLM` is the model an agent thinks with. Tasks run one after another, in the order you list them.

In [3]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool

# Step 4: Add API keys

> **Note:** The OpenAI key comes from **Colab Secrets** (key icon in the left sidebar, add a secret named `openai`, switch on **Notebook access**). The Serper key (Google search, https://serper.dev) is set directly in the notebook as `SERPER_API_KEY`, which is the name the search tool looks for.

In [16]:
import os
from google.colab import userdata

# OpenAI key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Serper key (Google search tool)

os.environ["SERPER_API_KEY"] = "b625f51914efbba89fd2b138127ac5c426003dba"

# Step 5: Set up the reasoning model

> **Note:** We use **one model for the whole crew**: OpenAI's reasoning model `gpt-5-mini`. A reasoning model thinks step by step before it answers, which gives better plans and better articles. `reasoning_effort` controls how long it thinks (`low`, `medium`, `high`). If OpenAI releases a newer model, change the one model name below.

In [5]:
llm = LLM(model="gpt-5-mini", reasoning_effort="medium")

# Step 6: Test the model

> **Note:** Before building agents, confirm the model answers. This catches a wrong key, a model name typo or missing billing in 10 seconds instead of in the middle of a crew run.

In [6]:
print(llm.call("Reply with exactly: reasoning model ready"))

reasoning model ready


# Step 7: Set up the tool (web search)

> **Note:** Tools let an agent act in the real world. `SerperDevTool` searches Google so the Planner can find fresh news. Only the Planner gets the tool, because research is the Planner's job. The Writer works from the Planner's output.

In [17]:
search_tool = SerperDevTool()

# Step 8: Create the Planner agent (reasoning model)

> **Note:** Every agent has a `role`, a `goal` and a `backstory`. LLMs perform better when they role play. `{topic}` is a placeholder that is filled in when we run the crew. The Planner uses the **reasoning model** and the **search tool**.
>
> Tip: several short strings in a row (instead of one triple-quoted string) avoid extra spaces and new lines being sent to the model.

In [18]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}. "
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    tools=[search_tool],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

# Step 9: Create the Writer agent

> **Note:** The Writer turns the Planner's outline into a 1500-word opinion piece. The backstory tells it to separate **facts** from **opinions**, which keeps the article honest. It uses the same reasoning model.

In [9]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic in 1500 words: {topic}",
    backstory="You're working on writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provided by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provided by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

# Step 10: Create the Editor agent

> **Note:** The Editor is the quality gate. It reviews the Writer's draft for grammar, balance and tone before anything is published. Separating writing from editing is the same reason real newsrooms have both roles.

In [10]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization.",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices, "
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

# Step 11: Create the Plan task

> **Note:** A task has a `description` (what to do), an `expected_output` (what done looks like) and an `agent` (who does it). A precise `expected_output` is the single biggest lever for output quality.

In [19]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

# Step 12: Create the Write task

> **Note:** We inject today's date so the Writer does not treat old information as current. The Write task automatically receives the Plan task's output as context, because tasks in a sequential crew pass their results forward.

In [12]:
from datetime import date
current_date = date.today().strftime("%B %d, %Y")
print("Today's date:", current_date)

write = Task(
    description=(
        f"Use today's date: {current_date}\n\n"
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs. "
        "About 1500 words.",
    agent=writer,
)

Today's date: November 21, 2026


# Step 13: Create the Edit task

> **Note:** The last task receives the draft and returns the publish-ready version. This final output is what the user sees.

In [21]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor,
)

# Step 14: Assemble the Crew

> **Note:** The crew connects agents and tasks. Order matters because tasks run one after another: Plan, then Write, then Edit. `verbose=True` prints every thought and action so learners can follow what each agent does.

In [20]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True,
)

# Step 15: Run the Crew (async)

> **Note:** Colab already runs an event loop, so a normal `crew.kickoff()` fails with *"Agent execution was invoked synchronously from within a running event loop"*. In a notebook we use the async version and put `await` in front of it. LLMs can give different output for the same input, so your article will differ from the demo. A full run usually takes 1 to 3 minutes.

In [22]:
topic = "Future of Super Intelligence"
result = await crew.kickoff_async(inputs={"topic": topic})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97574aa6-832e-49bf-abdf-9162028db5e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Future of Super Intelligence.       │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: eb90e853-902b-4028-8f06-1ec22af6d4f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Future of Super Intelligence.       │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'Future of superintelligence latest trends 2026 news key players OpenAI DeepMind        │
│  Anthropic Microsoft Google 2026 regulation AI safety AGI superintelligence developments 2026'}                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Output: {'searchParameters': {'q': 'Future of superintelligence latest trends 2026 news key players OpenAI     │
│  DeepMind Anthropic Microsoft Google 2026 regulation AI safety AGI superintelligence developments 2026',        │
│  'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'AI Safety Index — Summer 2026',       │
│  'link': 'https://futureoflife.org/ai-safety-index-summer-2026/', 'snippet': 'The Summer 2026 Index evaluates   │
│  nine leading AI companies on 37 indicators spanning six critical domains. The nine companies include           │
│  Anthropic, OpenAI, Google ...', 'position': 1}, {'title': '#Gravitas | The AI revolution faces its darkest     │
│  warning yet. Leaders from ...', 'link':                                                                        │
│  'https://www.facebook.com/WIONews/videos/gravitas-the-ai-revolution-faces-its-darkest-warning-yet-leaders-fro  │
│  m-openai-ant/2075044919775249/', 'snippet': '', 'position': 2}, {'title': 'From AGI to ASI', 'link':           │
│  'https://deepmind.google/research/publications/239142/', 'snippet': 'This report investigates how AI itself    │
│  might continue to develop in a post-AGI world along the continuum of machine intelligence.', 'position': 3},   │
│  {'title': 'Who Is Closest to Superintelligence in 2026? OpenAI vs Anthropic ...', 'link':                      │
│  'https://www.youtube.com/watch?v=Uum4pwq_VvI', 'snippet': 'This video explores the future of AI, AGI,          │
│  frontier models, AI safety, OpenAI GPT, Anthropic Claude, Google Gemini, xAI Grok, and the race', 'position':  │
│  4}, {'title': 'AI 2027: What Superintelligence Looks Like', 'link':                                            │
│  'https://www.lesswrong.com/posts/TpSFoqoG2M5MAAesg/ai-2027-what-superintelligence-looks-like-1', 'snippet':    │
│  'The stock market has gone up 30% in 2026, led by OpenBrain, Nvidia, and whichever companies have most         │
│  successfully integrated AI assistants.', 'position': 5}, {'title': 'The 2026 AI Safety Index: Nine AI Labs     │
│  Ranked by Safety and Security', 'link':                                                                        │
│  'https://dcthemedian.substack.com/p/the-2026-ai-safety-index-nine-ai', 'snippet': 'The summer 2026 scorecard   │
│  shows a clear separation at the top, with Anthropic, OpenAI, and Google DeepMind leading by a significant      │
│  margin.', 'position': 6}, {'title': 'Important trends in AI so far in 2026', 'link':                           │
│  'https://www.youtube.com/watch?v=OgKKD1tFD3M', 'snippet': 'AI is now everywhere and getting more autonomous.   │
│  We explore the major trends shaping enterprise adoption, legal risk, and the next wave of AI\xa0...',          │
│  'position': 7}, {'title': 'State of AI Report 2026 | AI Research, Industry and Policy', 'link':                │
│  'https://www.stateof.ai/', 'snippet': "The independent annual State of AI Report covering AI research,         │
│  industry, geopolitics and safety. Read the 2026 report, watch the author's cut, ...", 'position': 8},          │
│  {'title': 'Will we have AGI by 2030?', 'link': 'https://80000hours.org/ai/guide/when-will-agi-arrive/',        │
│  'snippet': 'Evidence-led look at why AGI might be feasible by 2030, the trends pushing capabilities forward,   │
│  and where they could plateau.', 'position': 9}, {'title': 'The Best AI Books in 2026', 'link':                 │
│  'https://fivebooks.com/best-books/the-best-ai-books-in

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AI regulation 2026 EU AI Act update 2026 US AI executive order 2026 AI governance      │
│  2026 G7 AI 2026 summit'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Future of Superintelligence: How Close We Are, What’s at Risk, and What Leaders Should Do Now            │
│  November 21, 2026                                                                                              │
│                                                                                                                 │
│  ## Introduction — Why this moment matters                                                                      │
│  The trajectory of advanced AI in 2026 makes the “future of superintelligence” an urgent policy and business    │
│  question rather than a distant thought experiment. Recent syntheses—notably the State of AI Report 2026 and    │
│  the AI Safety Index (Summer 2026)—document rapid capability gains in multimodal foundation models, widespread  │
│  deployment of agentic systems, and an uptick in public safety reporting. These signals, together with faster   │
│  regulatory activity such as the EU AI Act rollout and growing adoption of NIST-aligned guidance in the U.S.,   │
│  mean leaders must treat preparations for high-capability AI as a current priority.                             │
│                                                                                                                 │
│  This article synthesizes those sources and separates measured findings from judgments where appropriate.       │
│  Where assessments of likelihoods or policy preferences are offered, they are presented as reasoned             │
│  recommendations; where the piece summarizes published rankings or regulatory milestones, it relies on the      │
│  cited reports and public documentation. The goal is to map plausible pathways to superintelligence, explain    │
│  principal risks and trends shaping that trajectory, and offer concrete steps for policymakers, enterprises,    │
│  researchers, and civil society.                                                                                │
│                                                                                                                 │
│  ## Quick primer: What is superintelligence and why it’s different                                              │
│  “Superintelligence” (often called Artificial Superintelligence or ASI) describes an intelligence that          │
│  substantially exceeds human performance across most economically and scientifically valuable domains and that  │
│  can improve its own capabilities. It sits beyond the stage typically described as Artificial General           │
│  Intelligence (AGI), which denotes broad competence across diverse tasks. Rather than a single benchmark,       │
│  these concepts are best understood as a constellation of capabilities: cross-domain problem solving,           │
│  long‑horizon planning, tool use, and potential for self‑modification.                                          │
│                                                                                                                 │
│  Technical pathways that could lead toward superintelligence include continued scaling of foundation models,    │
│  integration of multimodal inputs (text, vision, audio, code), widespread agentization (persistent,             │
│  goal-directed agents), and theoretical mechanisms such as recursive self‑improvement. None of these paths      │
│  guarantees an automatic leap to ASI; architecture chan

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Step 16: Display the final article

> **Note:** `result.raw` is the final text produced by the last task (the Editor). Rendering it as Markdown shows headings and formatting exactly as it would appear on a blog.

In [23]:
from IPython.display import Markdown
Markdown(result.raw)

# The Future of Superintelligence: How Close We Are, What’s at Risk, and What Leaders Should Do Now  
November 21, 2026

## Introduction — Why this moment matters
The trajectory of advanced AI in 2026 makes the “future of superintelligence” an urgent policy and business question rather than a distant thought experiment. Recent syntheses—notably the State of AI Report 2026 and the AI Safety Index (Summer 2026)—document rapid capability gains in multimodal foundation models, widespread deployment of agentic systems, and an uptick in public safety reporting. These signals, together with faster regulatory activity such as the EU AI Act rollout and growing adoption of NIST-aligned guidance in the U.S., mean leaders must treat preparations for high-capability AI as a current priority.

This article synthesizes those sources and separates measured findings from judgments where appropriate. Where assessments of likelihoods or policy preferences are offered, they are presented as reasoned recommendations; where the piece summarizes published rankings or regulatory milestones, it relies on the cited reports and public documentation. The goal is to map plausible pathways to superintelligence, explain principal risks and trends shaping that trajectory, and offer concrete steps for policymakers, enterprises, researchers, and civil society.

## Quick primer: What is superintelligence and why it’s different
“Superintelligence” (often called Artificial Superintelligence or ASI) describes an intelligence that substantially exceeds human performance across most economically and scientifically valuable domains and that can improve its own capabilities. It sits beyond the stage typically described as Artificial General Intelligence (AGI), which denotes broad competence across diverse tasks. Rather than a single benchmark, these concepts are best understood as a constellation of capabilities: cross-domain problem solving, long‑horizon planning, tool use, and potential for self‑modification.

Technical pathways that could lead toward superintelligence include continued scaling of foundation models, integration of multimodal inputs (text, vision, audio, code), widespread agentization (persistent, goal-directed agents), and theoretical mechanisms such as recursive self‑improvement. None of these paths guarantees an automatic leap to ASI; architecture changes, compute constraints, funding flows, and breakthroughs in alignment research can accelerate or delay progress. Timelines are therefore probabilistic, not deterministic.

## Where we are in 2026 — capability milestones and structural facts
By late 2026 the field shows concrete capability and structural changes that make more rapid transitions plausible. Multimodal foundation models are mainstream in both research and products; production-ready agent stacks coordinate tools, APIs, and planning loops at scale; and labs increasingly experiment with long-horizon, self-supervised objectives. For summaries and data, see the State of AI Report 2026 (https://www.stateof.ai/) and the AI Safety Index (https://futureoflife.org/ai-safety-index-summer-2026/), which document capability growth alongside evolving safety practices.

A second structural fact is concentration: a handful of labs and cloud/hardware providers—OpenAI, Google DeepMind/Gemini, Anthropic, Microsoft Azure, NVIDIA, and Meta—control much of the compute and infrastructure needed for frontier models. That concentration lowers friction for large capability gains but increases systemic risk if incentives misalign. DeepMind’s work on post‑AGI trajectories is a useful reference for thinking through how incremental advances could aggregate into larger transitions (https://deepmind.google/research/publications/239142/).

## Key trends shaping the path to superintelligence
Agentic and autonomous systems have shifted from laboratory demos to production-ready stacks. Agents that compose tools, plan over multiple steps, and act autonomously introduce new failure modes—goal misalignment, unintended action chains, and emergent behaviors—not captured by single-query model testing. Organizations deploying these systems must treat them like critical infrastructure rather than simple APIs, with appropriate controls and incident-response plans.

Multimodality and foundation-model proliferation amplify capabilities across domains. When language, vision, audio, and code models converge, an AI that can read a paper, run experiments in simulation, write and debug code, and interpret results becomes qualitatively more powerful. This convergence not only increases productivity potential but also heightens the need for rigorous pre-deployment evaluation and domain-specific safety checks.

Safety and alignment have moved into the mainstream, and regulation is accelerating. Independent benchmarks, public red‑teaming, and safety-index reporting now shape reputations and funding. At the same time, the EU AI Act has entered enforcement phases and NIST guidance is influencing procurement and enterprise practice in the U.S., producing a mixed landscape of mandatory and voluntary standards that firms must navigate.

## Risks and realistic scenarios (short-, medium-, and long-term)
Short-term risks are the most likely and already observable: large-scale misinformation, automated cyber campaigns, the misuse of generative models to design hazardous biological sequences, and accidents from agentic systems performing high-stakes automation. These harms are addressable through well-established risk‑management tools—improved testing, incident reporting, human oversight, and operational safeguards—if organizations adopt them systematically.

Systemic risks emerge when capability concentration, automated control of critical infrastructure, and weak governance combine. Cascading failures in finance, supply chains, or other interdependent systems caused by poorly constrained agents could propagate rapidly. Reducing correlated failure modes requires transparency about compute and supply chains, enforceable pre-deployment checks, and resilient incident-response ecosystems.

Longer‑term, lower‑probability but high‑impact scenarios involve transitions to ASI and contested takeoff dynamics. Researchers debate “slow” versus “fast” takeoff: a slow takeoff offers more time for societal adaptation; a fast takeoff could present abrupt, hard-to-manage transitions. Given the uncertainty, a prudent strategy focuses on reducing tail risks now while enabling responsible deployments that fund and support safety work.

## What labs, governments, and industry are doing today
Leading labs increasingly adopt staged release policies, publish red‑teaming results, and maintain dedicated alignment teams—practices tracked by the AI Safety Index. Some firms publish safety metrics and submit to third‑party audits; others remain comparatively opaque. The resulting heterogeneity underscores the value of consistent audit standards and independent verification mechanisms.

Governments and standards bodies are responding with a mix of regulation and guidance. The EU AI Act and national sandbox programs create binding rules and experimental spaces for high‑risk systems, while NIST’s AI Risk Management Framework (https://www.nist.gov/itl/ai-risk-management-framework) has become a de facto baseline for U.S. procurement and enterprise practice. Industry meanwhile is developing governance tools—model cards, vendor audits, insurance pilots, and internal red teams—but quality and coverage vary widely.

## Practical recommendations — what each stakeholder should do now
For policymakers: adopt risk‑based rules that mandate pre‑deployment assessments for frontier, agentic, or dual‑use systems; fund independent auditing and verification capacity; require incident reporting for high‑risk releases; and invest in international coordination mechanisms. Near‑term priorities (6–12 months) should include establishing compute‑transparency standards and clarifying enforcement pathways under the EU AI Act and similar regimes.

For enterprise leaders: create an AI governance board with legal, security, and product representation; require vendor model cards and deployment‑impact assessments; mandate red‑team testing for agentic systems; adopt incident‑response and rollback procedures; and follow the NIST RMF as a practical baseline. Emphasize human‑in‑the‑loop designs where safety margins are narrow and document rationale for deployment decisions.

For researchers and labs: increase support for alignment research, publish red‑team findings when safe and feasible, and coordinate capability‑disclosure protocols to avoid dangerous surprise releases. For investors: make regulatory exposure and safety posture core due‑diligence criteria, and diversify infrastructure bets across cloud, hardware, and emerging safety tool providers. For civil society and the public: advocate for transparency and independent oversight, support reskilling programs for AI‑augmented work, and demand clearer incident reporting from both private and public actors.

## Scenarios and recommended policy packages
Three policy mixes merit comparison. “Precaution‑first” emphasizes tight controls and slow deployment; it reduces near‑term risk but can stifle beneficial innovation and push activity to jurisdictions with weaker rules. “Regulated innovation” pairs risk‑based rules, mandatory audits, and enforcement to preserve innovation while managing harms; this approach balances safety and growth. “Market‑led with safety incentives” relies on industry standards and liability regimes to motivate safer behavior but risks under‑enforcement for tail events if markets fail to internalize systemic risk.

For democratic governments seeking both growth and protection, the recommended package is “regulated innovation”: enforceable standards for frontier systems, investments in verification and auditing capacity, and coordinated international mechanisms to limit regulatory arbitrage. This mix aims to maximize societal benefits while constraining worst‑case outcomes.

## Resources and further reading
Key sources for ongoing monitoring and deeper study include the State of AI Report 2026 (https://www.stateof.ai/), the AI Safety Index (https://futureoflife.org/ai-safety-index-summer-2026/), DeepMind’s research on post‑AGI trajectories (https://deepmind.google/research/publications/239142/), the EU Artificial Intelligence Act resources (https://artificialintelligenceact.eu/), and NIST’s AI Risk Management Framework (https://www.nist.gov/itl/ai-risk-management-framework). For discussions about AGI timing and probabilistic forecasting, see 80,000 Hours’ AGI guide (https://80000hours.org/ai/guide/when-will-agi-arrive/).

These sources provide data, benchmarks, and policy analysis that practitioners can use to build governance frameworks, evaluate vendor claims, and prioritize safety investments. Bookmarking a handful of these reports and checking quarterly updates—especially new AI Safety Index releases and regulatory milestones—will help organizations stay current.

## Conclusion and call to action
The future of superintelligence is both a technical question and a social choice. Evidence in 2026 points to accelerating capabilities and stronger safety norms, alongside material gaps in governance and transparency. Decisive, coordinated action now—risk‑based regulation, clearer incentives for safety, and robust enterprise governance—meaningfully improves the odds of capturing benefits while limiting harms.

Take three concrete steps this month: policymakers should publish a timeline for compute‑transparency standards and incident‑reporting rules; enterprise leaders should establish or test an AI governance board and a red‑teaming process; and researchers and funders should prioritize reproducible alignment research and capability‑disclosure protocols. To support that work, download the Superintelligence Preparedness Checklist, sign up for our briefing series, or join the next policy consultation to help shape practical, enforceable standards.

# Step 17: Deploy the crew as a Gradio app

> **Note:** This is the step that turns a notebook into a **product**. We wrap the crew in one async function, `run_crew(topic)`, and give it a UI. Gradio supports async functions, so we use `await crew.kickoff_async(...)` here too. `share=True` creates a **public link** that works on any phone or laptop for about one week, with no server to set up. Anyone with the link can type a topic and get an article written by your agent team.
>
> **Be careful:** the app runs on **your** OpenAI and Serper keys, so every click costs you a small amount. Share the link only with people you trust, and stop the cell when the demo is over. For a permanent version, deploy to Hugging Face Spaces (see the live demo link at the top).

In [24]:
import gradio as gr

async def run_crew(topic_text):
    if not topic_text or not topic_text.strip():
        return "Please enter a topic first."
    try:
        output = await crew.kickoff_async(inputs={"topic": topic_text.strip()})
        return output.raw
    except Exception as e:
        return f"Something went wrong: {e}"

with gr.Blocks(title="From Prompt to Post") as demo:
    gr.Markdown(
        "# From Prompt to Post\n"
        "Enter a topic. A **Planner**, a **Writer** and an **Editor** powered by a reasoning model "
        "will research, write and edit a full article for you. This takes 1 to 3 minutes."
    )
    topic_box = gr.Textbox(label="Topic", placeholder="e.g. The future of AI agents in healthcare", lines=2)
    run_btn = gr.Button("Generate article", variant="primary")
    article = gr.Markdown(label="Article")
    gr.Examples(
        examples=[["Who will win the FIFA World Cup 2026"],
                  ["How AI agents are changing customer support"],
                  ["Is Dubai becoming a global AI hub?"]],
        inputs=topic_box,
    )
    run_btn.click(run_crew, inputs=topic_box, outputs=article)

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c4f02d2ae5887acd9a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97574aa6-832e-49bf-abdf-9162028db5e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Future of Agentic AI and loss of    │
│  jobs and humanity.                                                                                             │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: eb90e853-902b-4028-8f06-1ec22af6d4f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Future of Agentic AI and loss of    │
│  jobs and humanity.                                                                                             │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': "agentic AI 2026 news agentic autonomous AI 'agentic' 2026 developments"}               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': "AI job losses 2026 automation layoffs study 2026 report 'job displacement' 'AI'        │
│  2026"}                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (44 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Agentic AI Is Here: How Autonomous Agents Will Reshape Jobs, Business, and Human Agency                      │
│  November 21, 2026                                                                                              │
│                                                                                                                 │
│  Agentic AI — autonomous, goal‑directed systems that plan, use tools and act across systems with limited human  │
│  supervision — is moving from research labs into enterprise production. Picture an agent that manages           │
│  procurement end‑to‑end: it compares suppliers, negotiates standardized contract terms within approved          │
│  guidelines, submits purchase orders and escalates exceptions to legal or finance teams. These scenarios are    │
│  already in pilot and early production in finance, IT operations, customer service and procurement.             │
│                                                                                                                 │
│  The stakes are significant. Agentic systems can unlock efficiency and enable new capabilities, but they also   │
│  accelerate workforce reshaping and surface ethical questions about agency, responsibility and dignity. This    │
│  article explains what agentic AI means in practice, who’s driving adoption, what evidence and forecasts say    │
│  about job impacts, how regulation is evolving (notably the EU AI Act), and concrete steps leaders and workers  │
│  should take now.                                                                                               │
│                                                                                                                 │
│  ## What is agentic AI? A quick primer on AI agents and autonomous AI agents                                    │
│  Agentic AI refers to systems that are explicitly goal‑directed and capable of chaining actions across tools    │
│  and interfaces with limited human step‑by‑step control. In practical terms an agent can plan a sequence of     │
│  tasks, invoke APIs or enterprise tools, consult stored knowledge or databases, draft communications, and       │
│  iterate on outcomes until goals are reached or exceptions require human intervention. This contrasts with      │
│  many generative AI assistants that primarily respond to prompts rather than persistently act in workflows.     │
│                                                                                                                 │
│  Technically, agentic systems combine large foundational models (LLMs and multimodal models) with               │
│  orchestration layers, tool‑invocation frameworks and stateful memory. Developer ecosystems such as LangChain   │
│  and AutoGen‑style frameworks accelerate agent development and integration into MLOps and RPA pipelines, which  │
│  is why “agentic AI” is best understood as an architecture: model + toolkit + orchestration + governance. That  │
│  architecture changes implementation tradeoffs — from evaluation and monitoring to responsibility and           │
│  compliance — compared with single‑turn generative features.                                                    │
│                                                                                                                 │
│  ## Current landscape and key players driving adoption 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 97574aa6-832e-49bf-abdf-9162028db5e1                                                                       │
│  Final Output: # Agentic AI Is Here: How Autonomous Agents Will Reshape Jobs, Business, and Human Agency        │
│  November 21, 2026                                                                                              │
│                                                                                                                 │
│  Agentic AI — autonomous, goal‑directed systems that plan, use tools and act across systems with limited human  │
│  supervision — is moving from research labs into enterprise production. Picture an agent that manages           │
│  procurement end‑to‑end: it compares suppliers, negotiates standardized contract terms within approved          │
│  guidelines, submits purchase orders and escalates exceptions to legal or finance teams. These scenarios are    │
│  already in pilot and early production in finance, IT operations, customer service and procurement.             │
│                                                                                                                 │
│  The stakes are significant. Agentic systems can unlock efficiency and enable new capabilities, but they also   │
│  accelerate workforce reshaping and surface ethical questions about agency, responsibility and dignity. This    │
│  article explains what agentic AI means in practice, who’s driving adoption, what evidence and forecasts say    │
│  about job impacts, how regulation is evolving (notably the EU AI Act), and concrete steps leaders and workers  │
│  should take now.                                                                                               │
│                                                                                                                 │
│  ## What is agentic AI? A quick primer on AI agents and autonomous AI agents                                    │
│  Agentic AI refers to systems that are explicitly goal‑directed and capable of chaining actions across tools    │
│  and interfaces with limited human step‑by‑step control. In practical terms an agent can plan a sequence of     │
│  tasks, invoke APIs or enterprise tools, consult stored knowledge or databases, draft communications, and       │
│  iterate on outcomes until goals are reached or exceptions require human intervention. This contrasts with      │
│  many generative AI assistants that primarily respond to prompts rather than persistently act in workflows.     │
│                                                                                                                 │
│  Technically, agentic systems combine large foundational models (LLMs and multimodal models) with               │
│  orchestration layers, tool‑invocation frameworks and stateful memory. Developer ecosystems such as LangChain   │
│  and AutoGen‑style frameworks accelerate agent development and integration into MLOps and RPA pipelines, which  │
│  is why “agentic AI” is best understood as an architecture: model + toolkit + orchestration + governance. That  │
│  architecture changes implementation tradeoffs — from evaluation and monitoring to responsibility and           │
│  compliance — compared with single‑turn generative features.                                                    │
│                                                                                                                 │
│  ## Current landscape and key players driving adoption

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c4f02d2ae5887acd9a.gradio.live


# Step 18: Why this matters

> **Note:** Read this out loud at the end of the workshop.

**1. One prompt is not a process.** A single chat prompt gives you a single guess. A crew gives you a *workflow*: research, then write, then edit. Each agent does one job well, and the quality of the final output goes up.

**2. A reasoning model thinks before it writes.** Every agent in the crew uses a reasoning model, so plans are more careful and articles are better structured. You can tune how hard it thinks with `reasoning_effort`.

**3. Tools connect agents to the real world.** With web search, the Planner works with today's information, not only what the model memorised during training.

**4. Gradio turns code into a product.** A notebook only helps the person who can run it. A shareable app lets a manager, a client or a colleague use the same agents with no code at all. That is the jump from *experiment* to *something people use*.

**5. The pattern repeats everywhere.** Replace *Planner / Writer / Editor* with *Analyst / Reviewer / Reporter*, *Researcher / Drafter / Compliance checker*, or any workflow in your own company. The structure stays the same: **agents + tasks + tools + a UI**.

**Next steps:** try a different topic, change `reasoning_effort` to `"high"`, add a fourth agent (for example an SEO specialist), then deploy the app to Hugging Face Spaces for a permanent link.

# Step 19 Improvement in UI using AI

In [27]:
!pip install -q gradio==6.32.0

import asyncio
import os
import gradio as gr

# --- Custom CSS Design ---
css_code = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&family=Space+Grotesk:wght@500;600;700&display=swap');

body, .gradio-container {
    background-color: #05070B !important;
    font-family: 'Inter', sans-serif !important;
    color: #E2E8F0 !important;
    max-width: 1200px !important;
    margin: 0 auto !important;
}

h1, h2, h3, h4 {
    font-family: 'Space Grotesk', sans-serif !important;
    color: #F8FAFC !important;
}

.card-container {
    background-color: #0B1220 !important;
    border: 1px solid #22304A !important;
    border-radius: 4px !important;
    padding: 1.5rem !important;
    margin-bottom: 1rem !important;
}

.pipeline-card {
    background-color: #0B1220 !important;
    border: 1px solid #22304A !important;
    border-radius: 4px !important;
    padding: 1rem !important;
    text-align: center;
    transition: border-color 0.3s ease;
}

.pipeline-card.active {
    border-color: #FFB454 !important;
}

.mono-text {
    font-family: 'JetBrains Mono', monospace !important;
    color: #9FD6FF !important;
}

.logo-title-container {
    display: flex;
    align-items: center;
    gap: 1.5rem;
    padding: 1.5rem 0;
    border-bottom: 1px solid #22304A;
    margin-bottom: 2rem;
}

.logo-img {
    height: 56px;
    width: auto;
}

.title-group {
    display: flex;
    flex-direction: column;
}

.main-title {
    font-size: 1.75rem;
    font-weight: 700;
    margin: 0;
    color: #F8FAFC;
}

.main-subtitle {
    font-size: 0.95rem;
    color: #9FD6FF;
    margin: 0.25rem 0 0 0;
}

/* Override default Gradio components to respect borders and colors */
.gr-button-primary {
    background-color: #9FD6FF !important;
    color: #05070B !important;
    font-weight: 600 !important;
    border-radius: 4px !important;
    border: none !important;
}

.gr-button-primary:hover {
    background-color: #C2E7FF !important;
}

.gr-button-secondary {
    background-color: transparent !important;
    border: 1px solid #22304A !important;
    color: #9FD6FF !important;
    border-radius: 4px !important;
}

.gr-button-secondary:hover {
    background-color: #0B1220 !important;
}

textarea, input {
    background-color: #05070B !important;
    border: 1px solid #22304A !important;
    border-radius: 4px !important;
    color: #F8FAFC !important;
}

.tab-nav button {
    font-family: 'Space Grotesk', sans-serif !important;
    border-radius: 4px 4px 0 0 !important;
}

.tab-nav button.selected {
    color: #9FD6FF !important;
    border-color: #22304A !important;
    background-color: #0B1220 !important;
}

.markdown-rendered h1, .markdown-rendered h2, .markdown-rendered h3 {
    color: #9FD6FF !important;
    border-bottom: 1px solid #22304A;
    padding-bottom: 0.3em;
}

.markdown-rendered p {
    line-height: 1.7;
    margin-bottom: 1.25em;
    color: #E2E8F0;
}
"""

# --- Original logic wrapper & simulation generator ---
async def process_generation(topic_text):
    if not topic_text or not topic_text.strip():
        yield (
            "Please enter a topic first.",
            "",
            0,
            "Elapsed: 0 s",
            gr.HTML("<div style='color: #FFB454;'>Please provide a valid topic to begin.</div>"),
            gr.HTML("<div class='pipeline-card'>1 Planner<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Researches & outlines</span></div>"),
            gr.HTML("<div class='pipeline-card'>2 Writer<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Drafts ~1500 words</span></div>"),
            gr.HTML("<div class='pipeline-card'>3 Editor<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Polishes & checks</span></div>"),
            gr.Button(interactive=True)
        )
        return

    # Disable run button immediately
    yield (
        "Starting multi-agent system...",
        "",
        0,
        "Elapsed: 0 s",
        gr.HTML("<div class='mono-text'>Initializing crew agents... (estimated)</div>"),
        gr.HTML("<div class='pipeline-card active' style='border-color: #FFB454 !important;'>1 Planner<br/><span style='font-size:0.8rem;color:#FFB454;'>Researches & outlines</span></div>"),
        gr.HTML("<div class='pipeline-card'>2 Writer<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Drafts ~1500 words</span></div>"),
        gr.HTML("<div class='pipeline-card'>3 Editor<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Polishes & checks</span></div>"),
        gr.Button(interactive=False)
    )

    # Run crew in a background task to track actual execution status and live counter
    loop = asyncio.get_running_loop()
    crew_task = loop.create_task(crew.kickoff_async(inputs={"topic": topic_text.strip()}))

    start_time = asyncio.get_event_loop().time()
    progress = 0
    elapsed = 0

    while not crew_task.done():
        await asyncio.sleep(1)
        elapsed = int(asyncio.get_event_loop().time() - start_time)

        # Smooth asymptotic estimate of progress staying below 95%
        if progress < 90:
            progress += (90 - progress) * 0.05

        # Highlight pipeline steps based on estimated time thresholds
        if elapsed < 25:
            p1, p2, p3 = "active", "", ""
            p1_col, p2_col, p3_col = "#FFB454", "#9FD6FF", "#9FD6FF"
        elif elapsed < 75:
            p1, p2, p3 = "", "active", ""
            p1_col, p2_col, p3_col = "#9FD6FF", "#FFB454", "#9FD6FF"
        else:
            p1, p2, p3 = "", "", "active"
            p1_col, p2_col, p3_col = "#9FD6FF", "#9FD6FF", "#FFB454"

        status_text = f"Generating article... Running step {1 if elapsed < 25 else (2 if elapsed < 75 else 3)} (estimated)"

        yield (
            "",
            "",
            int(progress),
            f"Elapsed: {elapsed} s",
            gr.HTML(f"<div class='mono-text'>{status_text}</div>"),
            gr.HTML(f"<div class='pipeline-card {p1}'>1 Planner<br/><span style='font-size:0.8rem;color:{p1_col};'>Researches & outlines</span></div>"),
            gr.HTML(f"<div class='pipeline-card {p2}'>2 Writer<br/><span style='font-size:0.8rem;color:{p2_col};'>Drafts ~1500 words</span></div>"),
            gr.HTML(f"<div class='pipeline-card {p3}'>3 Editor<br/><span style='font-size:0.8rem;color:{p3_col};'>Polishes & checks</span></div>"),
            gr.Button(interactive=False)
        )

    try:
        output = await crew_task
        article_raw = output.raw
        word_count = len(article_raw.split())
        summary_msg = f"Completed in {elapsed} seconds. Word count: {word_count} words."

        # Save file to disk for downloading
        filename = "article.md"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(article_raw)

        yield (
            article_raw,
            article_raw,
            100,
            f"Elapsed: {elapsed} s",
            gr.HTML(f"<div class='mono-text' style='color:#9FD6FF;'>{summary_msg}</div>"),
            gr.HTML("<div class='pipeline-card'>1 Planner<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Completed</span></div>"),
            gr.HTML("<div class='pipeline-card'>2 Writer<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Completed</span></div>"),
            gr.HTML("<div class='pipeline-card'>3 Editor<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Completed</span></div>"),
            gr.Button(interactive=True)
        )
    except Exception as e:
        yield (
            f"An error occurred: {e}",
            "",
            0,
            "Elapsed: 0 s",
            gr.HTML(f"<div style='color: #FFB454;'>Something went wrong: {e}</div>"),
            gr.HTML("<div class='pipeline-card'>1 Planner<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Researches & outlines</span></div>"),
            gr.HTML("<div class='pipeline-card'>2 Writer<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Drafts ~1500 words</span></div>"),
            gr.HTML("<div class='pipeline-card'>3 Editor<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Polishes & checks</span></div>"),
            gr.Button(interactive=True)
        )

# --- File Download Helper ---
def handle_download():
    if os.path.exists("article.md"):
        return "article.md"
    return None

# --- Create Gradio App Interface ---
with gr.Blocks(title="From Prompt to Post") as demo:

    # Full Width Header
    gr.HTML("""
    <div class='logo-title-container'>
        <img class='logo-img' src='https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/DDS%20HIGH%20RES%20Logo%20%20copywhite.png' alt='DDS Logo' />
        <div class='title-group'>
            <h1 class='main-title'>From Prompt to Post</h1>
            <p class='main-subtitle'>Multi-agent writing assistant: plan, write, edit</p>
        </div>
    </div>
    """)

    with gr.Row():

        # Left Column (Wider, ~65%)
        with gr.Column(scale=65):

            # Card: Your Topic
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>Your topic</h3>")
                topic_box = gr.Textbox(
                    label="Topic",
                    placeholder="Enter a topic, e.g., The future of AI agents in healthcare",
                    lines=2,
                    show_label=False
                )
                run_btn = gr.Button("Generate article", variant="primary")

            # Card: Three-step Pipeline Strip
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>Pipeline status</h3>")
                with gr.Row():
                    step_1 = gr.HTML("<div class='pipeline-card'>1 Planner<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Researches & outlines</span></div>")
                    step_2 = gr.HTML("<div class='pipeline-card'>2 Writer<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Drafts ~1500 words</span></div>")
                    step_3 = gr.HTML("<div class='pipeline-card'>3 Editor<br/><span style='font-size:0.8rem;color:#9FD6FF;'>Polishes & checks</span></div>")

            # Card: Status Area
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>Execution Details</h3>")
                with gr.Row():
                    timer_txt = gr.Markdown("Elapsed: 0 s", elem_classes=["mono-text"])
                p_bar = gr.Slider(minimum=0, maximum=100, value=0, interactive=False, label="Overall Progress")
                status_area = gr.HTML("<div class='mono-text'>Ready to begin</div>")

            # Card: Result Area
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>Generated Output</h3>")
                with gr.Tabs():
                    with gr.TabItem("Article"):
                        article_md = gr.Markdown(elem_classes=["markdown-rendered"])
                    with gr.TabItem("Markdown source"):
                        article_source = gr.Code(language="markdown", interactive=False)

                download_btn = gr.Button("Download Article (.md)", variant="secondary")
                download_file = gr.File(label="Downloaded File", visible=False)

        # Right Column (Narrower, ~35%)
        with gr.Column(scale=35):

            # Card: Trending Topics
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>Trending topics</h3>")
                trend_1 = gr.Button("AI agents in the enterprise", size="sm", variant="secondary")
                trend_2 = gr.Button("Who will win the FIFA World Cup 2026", size="sm", variant="secondary")
                trend_3 = gr.Button("The future of remote work", size="sm", variant="secondary")
                trend_4 = gr.Button("Generative AI in healthcare", size="sm", variant="secondary")
                trend_5 = gr.Button("Cybersecurity in 2026", size="sm", variant="secondary")
                trend_6 = gr.Button("Electric vehicles and the energy transition", size="sm", variant="secondary")
                trend_7 = gr.Button("Dubai as a global AI hub", size="sm", variant="secondary")
                trend_8 = gr.Button("Personal finance for young professionals", size="sm", variant="secondary")

            # Card: How It Works
            with gr.Column(elem_classes=["card-container"]):
                gr.HTML("<h3>How it works</h3>")
                gr.HTML("""
                <div style='font-size:0.9rem; line-height:1.6; color:#E2E8F0;'>
                    1. Planner gathers real-time Google search context.<br/>
                    2. Writer creates structured, informative content.<br/>
                    3. Editor polishes grammar, style, and brand voice.
                </div>
                """)

    # --- Event Binding Setup ---
    trend_buttons = [trend_1, trend_2, trend_3, trend_4, trend_5, trend_6, trend_7, trend_8]
    for btn in trend_buttons:
        btn.click(lambda x=btn: x.value, outputs=topic_box)

    run_btn.click(
        process_generation,
        inputs=topic_box,
        outputs=[article_md, article_source, p_bar, timer_txt, status_area, step_1, step_2, step_3, run_btn]
    )

    download_btn.click(
        handle_download,
        outputs=download_file
    )

# --- Queue and Launch App Configuration (Gradio 6 style) ---
demo.queue()
demo.launch(share=True, debug=True, css=css_code, theme=gr.themes.Base())


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ada3e31b788322eb4f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97574aa6-832e-49bf-abdf-9162028db5e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on The future of remote work and       │
│  freelancing in age of AI.                                                                                      │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: eb90e853-902b-4028-8f06-1ec22af6d4f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on The future of remote work and       │
│  freelancing in age of AI.                                                                                      │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#14) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': "2025 trends remote work AI impact freelancing news 2024 2025 2026 report 'remote       │
│  work' 'AI' 'freelancing' trends"}                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#15) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': "Upwork Future Workforce Report 2024 2025 freelancing statistics 'Freelancing in        │
│  America' 2024 Upwork 2025 report"}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (29 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Future of Remote Work and Freelancing in the Age of AI: What You Need to Know                            │
│  November 21, 2026                                                                                              │
│                                                                                                                 │
│  Introduction                                                                                                   │
│                                                                                                                 │
│  Remote work and freelancing are now core components of the global labor market — and artificial intelligence   │
│  is reshaping what those careers look like. Marketplaces report continued growth in freelance earnings (Upwork  │
│  estimates ~$1.5T in recent freelance earnings), while research from OpenAI, Brookings, Gallup and others       │
│  documents rapid AI adoption across industries and early shifts in freelance demand ([Upwork Future Workforce   │
│  Index 2026](https://www.upwork.com/research/research-future-workforce-index-2026); [OpenAI                     │
│  research](https://openai.com/index/ai-effect-on-freelance-work/); [Brookings                                   │
│  analysis](https://www.brookings.edu/articles/is-generative-ai-a-job-killer-evidence-from-the-freelance-market  │
│  /)). The market is large, the tools are powerful, and the rules are changing.                                  │
│                                                                                                                 │
│  This article explains how AI is altering remote work and freelancing, who is likely to win or lose, which      │
│  skills and tools matter, what platforms and regulations to watch, and a practical action plan you can use      │
│  this quarter. Where assertions are opinionated, they are labeled as such; where possible, claims are backed    │
│  by public research and marketplace signals to help you make evidence-based decisions.                          │
│                                                                                                                 │
│  Where we are now: remote work and freelancing by the numbers                                                   │
│                                                                                                                 │
│  Remote and hybrid work remain mainstream even as some companies experiment with return-to-office policies.     │
│  Marketplaces continue to expand and diversify: Upwork and other platforms report rising shares of skilled      │
│  freelancing and more project-based budgets, while corporate adoption of AI tools is increasing the types of    │
│  remote work that can be delivered asynchronously and at scale ([Upwork                                         │
│  resources](https://www.upwork.com/resources/freelancing-stats);                                                │
│  [Gallup](https://www.gallup.com/workplace/704225/rising-adoption-spurs-workplace-changes.aspx)). Demand for    │
│  specialized, outcome-focused engagements remains robust across marketing, design, development and data roles.  │
│                                                                                                                 │
│  At the same time, academic and platform studies show m

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 97574aa6-832e-49bf-abdf-9162028db5e1                                                                       │
│  Final Output: # The Future of Remote Work and Freelancing in the Age of AI: What You Need to Know              │
│  November 21, 2026                                                                                              │
│                                                                                                                 │
│  Introduction                                                                                                   │
│                                                                                                                 │
│  Remote work and freelancing are now core components of the global labor market — and artificial intelligence   │
│  is reshaping what those careers look like. Marketplaces report continued growth in freelance earnings (Upwork  │
│  estimates ~$1.5T in recent freelance earnings), while research from OpenAI, Brookings, Gallup and others       │
│  documents rapid AI adoption across industries and early shifts in freelance demand ([Upwork Future Workforce   │
│  Index 2026](https://www.upwork.com/research/research-future-workforce-index-2026); [OpenAI                     │
│  research](https://openai.com/index/ai-effect-on-freelance-work/); [Brookings                                   │
│  analysis](https://www.brookings.edu/articles/is-generative-ai-a-job-killer-evidence-from-the-freelance-market  │
│  /)). The market is large, the tools are powerful, and the rules are changing.                                  │
│                                                                                                                 │
│  This article explains how AI is altering remote work and freelancing, who is likely to win or lose, which      │
│  skills and tools matter, what platforms and regulations to watch, and a practical action plan you can use      │
│  this quarter. Where assertions are opinionated, they are labeled as such; where possible, claims are backed    │
│  by public research and marketplace signals to help you make evidence-based decisions.                          │
│                                                                                                                 │
│  Where we are now: remote work and freelancing by the numbers                                                   │
│                                                                                                                 │
│  Remote and hybrid work remain mainstream even as some companies experiment with return-to-office policies.     │
│  Marketplaces continue to expand and diversify: Upwork and other platforms report rising shares of skilled      │
│  freelancing and more project-based budgets, while corporate adoption of AI tools is increasing the types of    │
│  remote work that can be delivered asynchronously and at scale ([Upwork                                         │
│  resources](https://www.upwork.com/resources/freelancing-stats);                                                │
│  [Gallup](https://www.gallup.com/workplace/704225/rising-adoption-spurs-workplace-changes.aspx)). Demand for    │
│  specialized, outcome-focused engagements remains robust across marketing, design, development and data roles.  │
│                                                                                                                 │
│  At the same time, academic and platform studies show 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ada3e31b788322eb4f.gradio.live
